## Task 1 — Arquitecturas modernas aplicadas al diagnóstico de enfermedades en hojas de mango

### 1. Por qué una red secuencial extremadamente profunda tipo VGG (150 capas) no sería una buena decisión para este problema y cómo ResNet resuelve la situación

Si se piensa en diseñar un modelo para detectar enfermedades en hojas de mango, una idea inicial podría ser construir una red convolucional extremadamente profunda bajo la intuición de que mayor profundidad implica mayor capacidad para aprender patrones complejos. En teoría esto es cierto, porque una red profunda puede aproximar funciones más complejas que una red poco profunda. Sin embargo, cuando la profundidad crece demasiado, empiezan a aparecer problemas serios de optimización que hacen que el entrenamiento se vuelva inestable o incluso contraproducente. En el caso de una arquitectura secuencial tipo VGG extendida a aproximadamente 150 capas, los principales problemas que aparecerían serían el desvanecimiento del gradiente y el fenómeno de degradación del entrenamiento.

El problema del desvanecimiento del gradiente aparece porque durante el entrenamiento utilizamos backpropagation para actualizar los parámetros del modelo. Si representamos la red como una composición de funciones

$$
x_L = f_L(f_{L-1}(...f_1(x)))
$$

entonces el gradiente que llega a las capas iniciales depende de la multiplicación de muchas derivadas a lo largo de la red

$$
\frac{\partial \mathcal{L}}{\partial x_1}
=
\frac{\partial \mathcal{L}}{\partial x_L}
\prod_{i=2}^{L}\frac{\partial x_i}{\partial x_{i-1}}
$$

Cuando las derivadas tienen valores menores que uno, este producto se vuelve muy pequeño conforme aumenta el número de capas. En una red con 150 capas, esto significa que las primeras capas prácticamente dejan de recibir señal de aprendizaje. En el contexto del problema de hojas de mango, esto es particularmente negativo porque las primeras capas de la red son las que aprenden a detectar patrones visuales básicos como bordes, texturas y pequeñas irregularidades en la superficie de la hoja. Si estas capas no se entrenan correctamente, la red pierde la capacidad de capturar detalles visuales importantes como pequeñas manchas, puntos de infección o cambios sutiles en la textura de la hoja.

Además del problema del gradiente, aparece el fenómeno conocido como degradación del entrenamiento. Este fenómeno consiste en que al agregar más capas a una red ya profunda, el error de entrenamiento comienza a aumentar en lugar de disminuir. Esto parece contradictorio porque una red más profunda debería poder representar al menos la misma función que una red más pequeña. Sin embargo, el problema es que una arquitectura secuencial tradicional no tiene ningún mecanismo que facilite aprender transformaciones identidad en las capas adicionales. Como resultado, el espacio de optimización se vuelve más difícil de explorar y el modelo puede terminar convergiendo a soluciones peores que una red más simple.

La arquitectura ResNet introduce una solución a este problema mediante el uso de conexiones residuales. En lugar de obligar a cada bloque a aprender directamente una transformación completa $H(x)$, el bloque aprende una función residual definida como

$$
F(x) = H(x) - x
$$

lo cual implica que la salida del bloque se puede escribir como

$$
H(x) = F(x) + x
$$

Este pequeño cambio modifica significativamente el comportamiento del entrenamiento. Si la transformación ideal es cercana a la identidad, el bloque únicamente necesita aprender una pequeña corrección sobre la entrada. En el caso del análisis de hojas de mango, esto permite que los bloques de la red aprendan refinamientos progresivos sobre las representaciones visuales de la hoja, en lugar de reconstruir completamente la información en cada capa.

Desde el punto de vista de backpropagation, las conexiones residuales también mejoran el flujo del gradiente. La derivada respecto a la entrada del bloque se convierte en

$$
\frac{\partial \mathcal{L}}{\partial x}
=
\frac{\partial \mathcal{L}}{\partial H}
\left(
\frac{\partial F}{\partial x} + 1
\right)
$$

La presencia del término $+1$ garantiza que siempre exista un camino directo para el gradiente hacia capas anteriores. Esto evita que el gradiente desaparezca completamente incluso cuando la red es muy profunda. En la práctica, esto permite entrenar redes de decenas o incluso cientos de capas sin que el proceso de aprendizaje se vuelva inestable.

Para un sistema de diagnóstico de enfermedades en hojas de mango, esto significa que podemos utilizar redes profundas que capturen patrones complejos en las hojas sin sacrificar estabilidad en el entrenamiento. La red puede aprender representaciones jerárquicas donde las primeras capas detectan texturas básicas de la hoja, las capas intermedias detectan patrones de infección y las capas profundas integran esa información para clasificar correctamente el tipo de enfermedad. Sin las conexiones residuales, entrenar una red tan profunda sería mucho más difícil y menos confiable.


### 2. Por qué la arquitectura Inception es especialmente adecuada para analizar enfermedades en hojas de mango

El análisis visual de enfermedades en hojas de mango presenta una dificultad particular: los patrones visuales asociados a las enfermedades aparecen en distintas escalas espaciales. Algunas infecciones generan pequeñas manchas oscuras distribuidas en diferentes puntos de la hoja, mientras que otras producen áreas grandes de decoloración o regiones completas cubiertas por moho. Esto significa que el modelo debe ser capaz de detectar tanto patrones muy locales como estructuras visuales más amplias.

En redes convolucionales, el tamaño del filtro determina el campo receptivo de las neuronas. Filtros pequeños como $3\times3$ capturan detalles finos y estructuras locales, mientras que filtros más grandes como $5\times5$ permiten capturar patrones más amplios en la imagen. Si utilizamos únicamente un tamaño de filtro en toda la red, el modelo queda limitado a analizar la imagen a una sola escala. Esto puede ser problemático cuando los patrones relevantes aparecen en diferentes tamaños dependiendo del tipo de enfermedad o del estado de desarrollo de la infección.

La arquitectura Inception aborda este problema aplicando múltiples filtros de distintos tamaños en paralelo dentro del mismo bloque. Matemáticamente, si $x$ representa la entrada al módulo, la salida puede representarse como

$$
y =
\text{concat}
\left(
f_{1\times1}(x),
f_{3\times3}(x),
f_{5\times5}(x),
f_{pool}(x)
\right)
$$

Cada una de estas ramas captura información a una escala diferente. En el contexto del análisis de hojas de mango, esto permite que la red detecte simultáneamente pequeñas lesiones puntuales, patrones de manchas medianas y áreas más extensas de infección. Esta capacidad de análisis multiescala es especialmente importante cuando las enfermedades pueden manifestarse de formas visuales muy distintas.

Sin embargo, aplicar múltiples ramas convolucionales en paralelo puede generar un aumento considerable en el número de parámetros y operaciones de cómputo. El costo aproximado de una convolución estándar puede expresarse como

$$
K^2 \cdot C_{in} \cdot C_{out} \cdot H \cdot W
$$

donde $K$ es el tamaño del kernel, $C_{in}$ el número de canales de entrada y $C_{out}$ el número de filtros. Si aplicamos filtros grandes directamente sobre tensores con muchos canales, el costo computacional puede crecer rápidamente.

Para evitar este problema, los módulos Inception utilizan convoluciones $1\times1$ antes de aplicar filtros más grandes. Estas convoluciones actúan como una reducción de dimensionalidad que disminuye el número de canales sobre los cuales operan las convoluciones más costosas. Por ejemplo, una entrada con 256 canales puede reducirse a 64 canales mediante una convolución $1\times1$ antes de aplicar un filtro $5\times5$.

Desde una perspectiva práctica, esta estrategia permite construir modelos capaces de capturar patrones visuales complejos en las hojas sin que el costo computacional crezca de manera descontrolada. En un sistema que eventualmente podría ejecutarse en dispositivos con recursos limitados, mantener este equilibrio entre expresividad del modelo y eficiencia computacional es fundamental.


### 3. Cómo funciona MobileNet y por qué su diseño es adecuado para ejecutar el modelo en teléfonos utilizados por agricultores

En el contexto del proyecto AgriTech, el sistema de diagnóstico no se ejecutará en servidores de alto rendimiento, sino directamente en teléfonos Android utilizados por agricultores en el campo. Esto introduce restricciones importantes relacionadas con memoria disponible, consumo energético y capacidad de procesamiento del dispositivo. En estas condiciones, arquitecturas diseñadas para servidores como ResNet o Inception pueden resultar demasiado pesadas para ejecutarse eficientemente.

La arquitectura MobileNet fue diseñada específicamente para este tipo de escenarios donde la eficiencia computacional es un factor crítico. Su principal innovación consiste en reemplazar las convoluciones estándar por una operación más eficiente llamada depthwise separable convolution. Esta operación separa el proceso de filtrado espacial del proceso de combinación entre canales.

En una convolución tradicional, cada filtro opera simultáneamente sobre todos los canales de entrada. El costo computacional de esta operación puede aproximarse como

$$
D_k^2 \cdot M \cdot N \cdot H \cdot W
$$

donde $D_k$ es el tamaño del kernel, $M$ el número de canales de entrada y $N$ el número de filtros.

MobileNet divide esta operación en dos pasos. Primero aplica una depthwise convolution que filtra espacialmente cada canal de manera independiente. El costo de esta operación es

$$
D_k^2 \cdot M \cdot H \cdot W
$$

Posteriormente aplica una pointwise convolution, que es una convolución $1\times1$ encargada de combinar la información entre canales. El costo de esta segunda operación es

$$
M \cdot N \cdot H \cdot W
$$

El costo total de la operación es entonces

$$
D_k^2 \cdot M \cdot H \cdot W + M \cdot N \cdot H \cdot W
$$

lo cual representa una reducción significativa en comparación con una convolución estándar.

El precio que se paga por esta eficiencia es una ligera reducción en la capacidad representacional del modelo. En una convolución estándar, cada filtro puede aprender simultáneamente relaciones espaciales complejas entre todos los canales de entrada. En MobileNet, estas dos operaciones se separan: primero se detectan patrones espaciales dentro de cada canal y luego se combinan los canales mediante una transformación lineal.

A pesar de esta limitación, el compromiso es razonable para una aplicación móvil de diagnóstico agrícola. El objetivo principal del sistema no es alcanzar la máxima precisión posible en un entorno de laboratorio, sino ofrecer una herramienta práctica que pueda ejecutarse directamente en el teléfono del agricultor. Un modelo que requiera conexión a internet o hardware especializado sería mucho menos útil en entornos rurales donde la conectividad puede ser limitada.

Por esta razón, MobileNet ofrece un equilibrio adecuado entre eficiencia computacional y capacidad de predicción. El modelo sigue siendo lo suficientemente expresivo para detectar patrones de enfermedad en las hojas, pero al mismo tiempo es lo suficientemente ligero para ejecutarse de forma rápida y eficiente en dispositivos móviles.